# MEMOIR Experiments — Session 1: Baselines + MEMOIR Main

## One-time setup (before your first session)

### Step 1 — Push code to GitHub
```bash
git add -A
git commit -m "add experiment scripts and kaggle notebooks"
git push
```

### Step 2 — Create a Kaggle dataset from preprocessed data
1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Name it exactly: `memoir-amazon-processed`
3. Upload from `data/processed/amazon/`:
   - `train_samples.parquet`, `val_samples.parquet`, `test_samples.parquet`
   - `windows.parquet`, `_metadata.json`
   - `embed_cache_TinyLlama_TinyLlama-1.1B-Chat-v1.0.pt` (optional, saves ~35 min)
4. Click **Create**

---

## Notebook settings
1. Accelerator → **GPU T4 x2**
2. Internet → **On**
3. **Add Data** → attach `memoir-amazon-processed`
4. Set `GITHUB_REPO` in Cell 2

## What this session runs
| Step | Contents | Est. time |
|------|----------|-----------|
| Baselines | GRU4Rec, SASRec, CL4SRec, DuoRec, SRA-CL, SASRec-Text, UniSRec | ~3 h |
| MEMOIR | Main model training | ~2 h |

After this session finishes, click **Save Version** so Session 2 can reuse the outputs.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Configuration
import os

GITHUB_REPO  = 'https://github.com/louiezzang/MEMOIR.git'
DATASET_NAME = 'datasets'  # adjust if your Kaggle dataset mount name differs

REPO_DIR   = '/kaggle/working/MEMOIR'
DATA_INPUT = f'/kaggle/input/{DATASET_NAME}'
WORK_DIR   = '/kaggle/working'
CKPT_DIR   = f'{WORK_DIR}/checkpoints'
LOG_DIR    = f'{WORK_DIR}/logs'

assert GITHUB_REPO, 'Set GITHUB_REPO'
if not os.path.exists(DATA_INPUT):
    # Auto-detect: find any input dir containing parquet files
    for name in os.listdir('/kaggle/input'):
        candidate = f'/kaggle/input/{name}'
        for root, dirs, files in os.walk(candidate):
            if any(f.endswith('.parquet') for f in files):
                DATASET_NAME = name
                DATA_INPUT = candidate
                break
        if os.path.exists(DATA_INPUT):
            break
assert os.path.exists(DATA_INPUT), f'Attach dataset. Check /kaggle/input/ contents: {os.listdir("/kaggle/input/")}'
print(f'Dataset: {DATASET_NAME}')
print(f'Dataset path: {DATA_INPUT}')

In [ ]:
# Cell 3: Clone / pull repo
import os
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
os.chdir(REPO_DIR)
# Clear bytecode cache to ensure latest code runs
os.system(f'find {REPO_DIR} -name "__pycache__" -exec rm -rf {{}} + 2>/dev/null; true')
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Cell 4: Install dependencies (preserve Kaggle's CUDA-enabled PyTorch)
import torch, subprocess
_torch_version = torch.__version__

!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    sentence-transformers>=3.0.0 \
    datasets>=2.19.0 \
    bitsandbytes \
    pandas pyarrow scikit-learn scipy pyyaml tqdm

# Check if pip replaced CUDA PyTorch (use subprocess to avoid reload issues)
check = subprocess.run(
    ['python', '-c', 'import torch; print(torch.cuda.is_available()); print(torch.__version__)'],
    capture_output=True, text=True
)
lines = check.stdout.strip().split('\n')
cuda_ok = lines[0] == 'True' if lines else False
new_version = lines[1] if len(lines) > 1 else 'unknown'

if not cuda_ok:
    print(f'WARNING: pip replaced CUDA PyTorch ({_torch_version}) with CPU build ({new_version})!')
    print('Reinstalling CUDA PyTorch...')
    subprocess.run(['pip', 'install', '-q', f'torch=={_torch_version}',
                    '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    check2 = subprocess.run(
        ['python', '-c', 'import torch; print(torch.cuda.is_available())'],
        capture_output=True, text=True
    )
    assert 'True' in check2.stdout, 'Failed to restore CUDA PyTorch!'
    print(f'Restored PyTorch {_torch_version} with CUDA.')
else:
    print(f'Done. PyTorch {new_version}, CUDA=True')

In [ ]:
# Cell 5: Link data & directories
import os, shutil

LOCAL_PROCESSED = f'{REPO_DIR}/data/processed/amazon'

# Find the directory containing the parquet files
data_source = None
for root, dirs, files in os.walk(DATA_INPUT):
    if any(f.endswith('.parquet') for f in files):
        data_source = root
        break

assert data_source, f'No parquet files found under {DATA_INPUT}'

# Ensure parent directory exists (data/processed/ is gitignored)
os.makedirs(os.path.dirname(LOCAL_PROCESSED), exist_ok=True)

# Copy data (not symlink) so MEMOIR can write if needed
if os.path.islink(LOCAL_PROCESSED):
    os.unlink(LOCAL_PROCESSED)
if os.path.exists(LOCAL_PROCESSED):
    shutil.rmtree(LOCAL_PROCESSED)

shutil.copytree(data_source, LOCAL_PROCESSED)

print(f'Data source: {data_source}')
print(f'Copied files: {os.listdir(LOCAL_PROCESSED)}')

# Verify critical files exist
for required in ['train_samples.parquet', 'val_samples.parquet', 'test_samples.parquet']:
    assert os.path.exists(f'{LOCAL_PROCESSED}/{required}'), f'Missing {required} after copy!'
print('All required parquet files verified.')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
local_ckpt = f'{REPO_DIR}/checkpoints'
local_log  = f'{REPO_DIR}/logs'
if not os.path.exists(local_ckpt):
    os.symlink(CKPT_DIR, local_ckpt)
if not os.path.exists(local_log):
    os.symlink(LOG_DIR, local_log)

print(f'Processed data: {os.listdir(LOCAL_PROCESSED)}')

In [ ]:
# Cell 6: Write Kaggle config (GPU-optimised)
import yaml, os

with open(f'{REPO_DIR}/configs/default.yaml') as f:
    config = yaml.safe_load(f)

config['data']['processed_dir']                  = f'{REPO_DIR}/data/processed'
config['model']['llm_load_in_4bit']              = False
config['model']['freeze_llm']                    = True
config['training']['batch_size']                 = 128
config['training']['gradient_accumulation_steps'] = 1
config['training']['fp16']                       = True
config['logging']['save_dir']                    = CKPT_DIR
config['logging']['log_dir']                     = LOG_DIR
config['logging']['save_every']                  = 5

KAGGLE_CONFIG = f'{REPO_DIR}/configs/kaggle.yaml'
with open(KAGGLE_CONFIG, 'w') as f:
    yaml.dump(config, f)

print('configs/kaggle.yaml written')
print(f"  processed_dir={config['data']['processed_dir']}")
print(f"  batch_size={config['training']['batch_size']}, fp16={config['training']['fp16']}")
print(f"  log_dir={LOG_DIR}")
print(f"  save_dir={CKPT_DIR}")

---
## Run Baselines + MEMOIR

In [ ]:
# Helper: run one baseline and show best result
import sys, json, subprocess
sys.path.insert(0, REPO_DIR)

SUBPROCESS_ENV = {**os.environ, 'PYTHONPATH': REPO_DIR, 'PYTHONDONTWRITEBYTECODE': '1'}

def run_baseline(model_name):
    log_dir = f'{LOG_DIR}/{model_name}_amazon'
    metrics_path = f'{log_dir}/metrics.json'

    # Skip if already done
    if os.path.exists(metrics_path):
        data = json.loads(open(metrics_path).read())
        evals = [x for x in data if x.get('type') == 'eval']
        if evals:
            best = max(evals, key=lambda x: x.get('ndcg@10', 0))
            print(f'[SKIP] {model_name} — already done '
                  f'(HR@10={best.get("hr@10",0):.4f}, NDCG@10={best.get("ndcg@10",0):.4f})')
            return

    print(f'[RUN ] {model_name}...')
    result = subprocess.run(
        ['python', 'baselines/train_baseline.py',
         '--config', KAGGLE_CONFIG,
         '--model', model_name,
         '--dataset', 'amazon'],
        capture_output=False, text=True,
        env=SUBPROCESS_ENV
    )
    if result.returncode != 0:
        print(f'[FAIL] {model_name} exited with code {result.returncode}')
        return

    if os.path.exists(metrics_path):
        data = json.loads(open(metrics_path).read())
        evals = [x for x in data if x.get('type') == 'eval']
        if evals:
            best = max(evals, key=lambda x: x.get('ndcg@10', 0))
            print(f'[DONE] {model_name} — '
                  f'HR@10={best.get("hr@10",0):.4f}, '
                  f'NDCG@10={best.get("ndcg@10",0):.4f}, '
                  f'HR@20={best.get("hr@20",0):.4f}, '
                  f'MRR={best.get("mrr",0):.4f}')

print('Helper ready.')

In [ ]:
# Run all baselines (~3 h total)
for model in ['gru4rec', 'sasrec', 'cl4srec', 'duorec', 'sracl', 'sasrec_text', 'unisrec']:
    run_baseline(model)

In [ ]:
# Train MEMOIR main (~2 h)
import subprocess

memoir_log = f'{LOG_DIR}/memoir_amazon'
memoir_metrics = f'{memoir_log}/metrics.json'

if os.path.exists(memoir_metrics):
    import json
    data = json.loads(open(memoir_metrics).read())
    evals = [x for x in data if x.get('type') == 'eval']
    if evals:
        best = max(evals, key=lambda x: x.get('ndcg@10', 0))
        print(f'[SKIP] MEMOIR main already done '
              f'(HR@10={best["hr@10"]:.4f}, NDCG@10={best["ndcg@10"]:.4f})')
    else:
        print('[INFO] metrics.json exists but empty — re-running')
        os.remove(memoir_metrics)
else:
    print('[RUN ] MEMOIR main...')
    subprocess.run(
        ['python', 'train.py', '--config', KAGGLE_CONFIG, '--dataset', 'amazon'],
        check=True,
        env=SUBPROCESS_ENV
    )

---
## Session 1 Results

In [ ]:
# Session 1 results summary
import json, os

MODELS = [
    ('GRU4Rec',       f'{LOG_DIR}/gru4rec_amazon'),
    ('SASRec',        f'{LOG_DIR}/sasrec_amazon'),
    ('CL4SRec',       f'{LOG_DIR}/cl4srec_amazon'),
    ('DuoRec',        f'{LOG_DIR}/duorec_amazon'),
    ('SRA-CL',        f'{LOG_DIR}/sracl_amazon'),
    ('SASRec-Text',   f'{LOG_DIR}/sasrec_text_amazon'),
    ('UniSRec',       f'{LOG_DIR}/unisrec_amazon'),
    ('MEMOIR (Ours)', f'{LOG_DIR}/memoir_amazon'),
]

def best_metrics(log_dir):
    p = f'{log_dir}/metrics.json'
    if not os.path.exists(p):
        return None
    data = json.loads(open(p).read())
    evals = [x for x in data if x.get('type') == 'eval']
    return max(evals, key=lambda x: x.get('ndcg@10', 0)) if evals else None

fmt = lambda v: f'{v:.4f}' if v is not None else '  --  '

print(f"{'Model':<22} {'HR@10':>7} {'NDCG@10':>9} {'HR@20':>7} {'MRR':>7}")
print('-' * 56)
for name, d in MODELS:
    m = best_metrics(d)
    if m:
        print(f"{name:<22} {fmt(m.get('hr@10')):>7} {fmt(m.get('ndcg@10')):>9} "
              f"{fmt(m.get('hr@20')):>7} {fmt(m.get('mrr')):>7}")
    else:
        print(f"{name:<22} {'(not run)':>34}")

print('\nSave Version to persist results for Session 2!')

---
## Next: Session 2

1. Click **Save Version** (top right) → *Save & Run All*
2. Open `memoir_kaggle_session2.ipynb`
3. **Add Data** → add this notebook's output as `memoir-session1-output`
4. Run all cells